# Factores determinantes del desempeño académico en las pruebas ICFES

In [ ]:

# Importar librerias

import pandas as pd
import numpy as np
from datetime import datetime
import requests
import pandas as pd
from google.colab import auth
from google.cloud import bigquery

In [ ]:
# Obtener los datos de API
base_url = "https://www.datos.gov.co/resource/rnvb-vnyh.json"
limit = 50000
offset = 0
all_data = []

while True:
    url = f"{base_url}?$limit={limit}&$offset={offset}"
    response = requests.get(url)
    data = response.json()
    if not data:
        break
    all_data.extend(data)
    offset += limit

df = pd.DataFrame(all_data)
df.head()

,estu_tipodocumento,estu_nacionalidad,estu_genero,estu_fechanacimiento,periodo,estu_consecutivo,estu_estudiante,estu_pais_reside,estu_tieneetnia,estu_depto_reside,...,punt_ingles,percentil_ingles,desemp_ingles,punt_global,percentil_global,estu_inse_individual,estu_nse_individual,estu_nse_establecimiento,estu_estadoinvestigacion,estu_generacion_e
0,TI,SUIZA,F,2003-03-03T00:00:00.000,20204,SB11202040211436,ESTUDIANTE,SUIZA,No,CUNDINAMARCA,...,55,81,A1,244,49,54.8823647351142,3,3,PUBLICAR,NO
1,PEP,VENEZUELA,M,2002-05-10T00:00:00.000,20204,SB11202040433216,ESTUDIANTE,VENEZUELA,No,CUNDINAMARCA,...,33,6,A-,238,44,49.2523109721028,2,2,PUBLICAR,NO
2,TI,VENEZUELA,F,2003-12-14T00:00:00.000,20204,SB11202040244180,ESTUDIANTE,VENEZUELA,No,CUNDINAMARCA,...,59,87,A2,325,94,40.7336723796904,1,3,PUBLICAR,GENERACION E - GRATUIDAD
3,CE,VENEZUELA,M,2003-04-12T00:00:00.000,20204,SB11202040210971,ESTUDIANTE,VENEZUELA,No,CUNDINAMARCA,...,47,58,A-,238,45,48.2179528165206,2,3,PUBLICAR,NO
4,TI,COLOMBIA,F,2004-03-03T00:00:00.000,20204,SB11202040235382,ESTUDIANTE,COLOMBIA,No,CUNDINAMARCA,...,43,40,A-,202,19,60.9121919624608,3,3,PUBLICAR,NO


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 504872 entries, 0 to 504871
Data columns (total 81 columns):
 #   Column                         Non-Null Count   Dtype 
---  ------                         --------------   ----- 
 0   estu_tipodocumento             504872 non-null  object
 1   estu_nacionalidad              504872 non-null  object
 2   estu_genero                    504864 non-null  object
 3   estu_fechanacimiento           504872 non-null  object
 4   periodo                        504872 non-null  object
 5   estu_consecutivo               504872 non-null  object
 6   estu_estudiante                504872 non-null  object
 7   estu_pais_reside               504872 non-null  object
 8   estu_tieneetnia                501879 non-null  object
 9   estu_depto_reside              504870 non-null  object
 10  estu_cod_reside_depto          504870 non-null  object
 11  estu_mcpio_reside              504870 non-null  object
 12  estu_cod_reside_mcpio          504870 non-nu

In [ ]:

# 3. CONVERSIÓN DE PUNTAJES A NUMÉRICO

COLS_PUNTAJE = [
    "punt_lectura_critica",
    "punt_matematicas",
    "punt_c_naturales",
    "punt_sociales_ciudadanas",
    "punt_ingles",
    "punt_global",
]

COLS_PERCENTIL = [
    "percentil_lectura_critica",
    "percentil_matematicas",
    "percentil_c_naturales",
    "percentil_sociales_ciudadanas",
    "percentil_ingles",
    "percentil_global",
]

for col in COLS_PUNTAJE + COLS_PERCENTIL:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:

# 2. LIMPIEZA GENERAL

def limpiar_strings(df):
    """Estandariza columnas de texto: mayúsculas, sin espacios extra."""
    cols_obj = df.select_dtypes(include="object").columns
    for col in cols_obj:
        df[col] = (df[col]
                   .astype(str)
                   .str.strip()
                   .str.upper()
                   .replace("NAN", np.nan))
    return df

df = limpiar_strings(df)

In [ ]:
 #Eliminar datos faltantes
df.dropna(inplace=True)

#Elimanar datos elimnados
df.drop_duplicates(inplace=True)

In [ ]:
#Almacenar datos en base de datos En nuestro caso utlizaremos almacén de datos de Google BIGQUERY

def autenticar_usuario():
    """Autentica al usuario en Google Colab."""
    auth.authenticate_user()
    print("Usuario autenticado correctamente.")


def inicializar_cliente(proyecto_id):
    """Inicializa el cliente de BigQuery con el proyecto especificado."""
    return bigquery.Client(project=proyecto_id)


def cargar_dataframe_a_bigquery(client, df, table_id, write_disposition="WRITE_TRUNCATE"):

    try:
        job_config = bigquery.LoadJobConfig(write_disposition=write_disposition)
        job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
        job.result()  # Esperar a que la carga se complete

        if job.errors:
            print(" Errores durante la carga:")
            for error in job.errors:
                print(error)
            return

        # Validar que la cantidad de filas coincida
        tabla = client.get_table(table_id)
        filas_df = df.shape[0]

        if tabla.num_rows == filas_df:
            print(f" Carga exitosa: {filas_df} filas cargadas correctamente a {table_id}.")
        else:
            print(f" Posible discrepancia: DataFrame tiene {filas_df} filas, pero la tabla tiene {tabla.num_rows}.")

    except Exception as e:
        print(f" Error al cargar los datos a BigQuery: {e}")

In [ ]:
# Paso 1: Autenticación
autenticar_usuario()

# Paso 2: Inicializar cliente
proyecto_id = "proyecto-etl-icfes"
cliente = inicializar_cliente(proyecto_id)

# Paso 3: Definir tabla y cargar DataFrame (df_interes ya debe existir)
tabla_destino = "proyecto-etl-icfes.PruebaEtl.prue_betl"
cargar_dataframe_a_bigquery(cliente, df, tabla_destino)

✅ Usuario autenticado correctamente.
 Carga exitosa: 371826 filas cargadas correctamente a proyecto-etl-icfes.PruebaEtl.prue_betl.
